# Hospital Appointment Data & RAG Assistant

**Modern Data Engineering for AI Systems: Final Project**

This notebook builds a small end-to-end hospital appointment project:

```text
Data Source → Spark → Clean & Transform → Delta Lake → Analytics
                                      ↓
                              Hospital FAQ → Simple RAG
```

**Problem:** hospital appointment data may contain missing values, duplicate appointments, invalid statuses, and incorrect waiting times.

We clean the data, store trusted data in Delta Lake, calculate simple hospital metrics, and use a small RAG component to answer appointment policy questions.

**Goal:** keep the code simple and easy to explain.

## Step 0 — Setup

Install Java, PySpark, and Delta Lake.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless
!pip install -q pyspark==3.5.3 delta-spark==3.3.2 rank_bm25

In [ ]:
import os
import glob
from pathlib import Path

java_homes = glob.glob("/usr/lib/jvm/java-17-openjdk-*")
os.environ["JAVA_HOME"] = java_homes[0]
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

BASE = Path("/content/hospital-appointment-project")
DATA_DIR = BASE / "data"
LAKEHOUSE_DIR = BASE / "lakehouse"

DATA_DIR.mkdir(parents=True, exist_ok=True)
LAKEHOUSE_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE)

## Step 1 — Create the Appointment Data

We create a small synthetic dataset.

The data includes:
- patient ID
- department
- appointment date
- appointment status
- waiting time

In [ ]:
import csv
import random
from datetime import date, timedelta

random.seed(42)

departments = ["Cardiology", "Dermatology", "Pediatrics", "Orthopedics"]
statuses = ["Completed", "Cancelled", "No Show"]

rows = []

for i in range(1, 101):
    patient_id = f"P{i:04d}"
    department = random.choice(departments)
    appointment_date = date(2026, 9, 1) + timedelta(days=random.randint(0, 29))
    status = random.choice(statuses)
    wait_time = random.randint(5, 60)

    rows.append([
        i,
        patient_id,
        department,
        appointment_date.isoformat(),
        status,
        wait_time
    ])

with open(DATA_DIR / "appointments.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "appointment_id",
        "patient_id",
        "department",
        "appointment_date",
        "status",
        "wait_time"
    ])
    writer.writerows(rows)

print("Created", len(rows), "appointments")

### Add a few data quality problems

We intentionally add a few errors so the cleaning step has something to find.

In [ ]:
with open(DATA_DIR / "appointments.csv", "a", newline="") as f:
    writer = csv.writer(f)

    # Missing patient ID
    writer.writerow([101, "", "Cardiology", "2026-09-15", "Completed", 20])

    # Invalid status
    writer.writerow([102, "P0102", "Dermatology", "2026-09-16", "Unknown", 15])

    # Invalid waiting time
    writer.writerow([103, "P0103", "Pediatrics", "2026-09-17", "Completed", -10])

    # Duplicate appointment ID
    writer.writerow([1, "P0001", "Cardiology", "2026-09-01", "Completed", 30])

print("Added test quality problems")

## Step 2 — Start Spark

Spark will read and process the appointment data.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("hospital-appointment-project")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

## Step 3 — Read the Raw Data

We first load the CSV as raw data.

At this stage, we do not try to fix the data yet.

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "appointments.csv"))
)

df.show(10)
print("Rows:", df.count())

## Step 4 — Data Quality Checks

We implement four simple quality checks:

1. **Completeness** — patient ID must not be missing.
2. **Validity** — appointment status must be valid.
3. **Accuracy / Business Rule** — waiting time cannot be negative.
4. **Uniqueness** — appointment ID must be unique.

The checks produce a quality result for each row.

In [ ]:
quality_df = df.withColumn(
    "completeness_check",
    F.col("patient_id").isNotNull() & (F.col("patient_id") != "")
)

quality_df = quality_df.withColumn(
    "validity_check",
    F.col("status").isin(statuses)
)

quality_df = quality_df.withColumn(
    "accuracy_check",
    F.col("wait_time") >= 0
)

quality_df = quality_df.withColumn(
    "uniqueness_check",
    F.lit(True)
)

quality_df.select(
    "appointment_id",
    "patient_id",
    "status",
    "wait_time",
    "completeness_check",
    "validity_check",
    "accuracy_check"
).show(10)

### Uniqueness Check

An appointment ID is valid only when it appears once in the batch.

In [ ]:
duplicate_ids = (
    df.groupBy("appointment_id")
      .count()
      .filter(F.col("count") > 1)
)

print("Duplicate appointment IDs:")
duplicate_ids.show()

print("Number of duplicate IDs:", duplicate_ids.count())

In [ ]:
quality_df = quality_df.withColumn(
    "uniqueness_check",
    ~F.col("appointment_id").isin(
        [row["appointment_id"] for row in duplicate_ids.collect()]
    )
)

quality_df = quality_df.withColumn(
    "quality_pass",
    F.col("completeness_check") &
    F.col("validity_check") &
    F.col("accuracy_check") &
    F.col("uniqueness_check")
)

quality_df.select(
    "appointment_id",
    "quality_pass"
).show(10)

## Step 5 — Quality Gate

The quality gate separates the batch into two paths:

```text
                 Quality Gate
                     |
              +------+------+
              |             |
            PASS           FAIL
              |             |
        Trusted Data    Quarantine
              |
          Delta Lake
```

**PASS:** all required checks are true.

**FAIL:** at least one quality check is false.

In [ ]:
passed_df = quality_df.filter(F.col("quality_pass") == True)
failed_df = quality_df.filter(F.col("quality_pass") == False)

print("PASS rows:", passed_df.count())
print("FAIL rows:", failed_df.count())

print("\nFailed rows:")
failed_df.select(
    "appointment_id",
    "patient_id",
    "status",
    "wait_time",
    "completeness_check",
    "validity_check",
    "accuracy_check",
    "uniqueness_check"
).show()

## Step 6 — Quarantine the Failed Data

Failed records do not continue to the trusted layer.

We store them separately in a **Quarantine** Delta table so they can be reviewed later.

In [ ]:
quarantine_path = str(LAKEHOUSE_DIR / "quarantine")

failed_df.write     .format("delta")     .mode("overwrite")     .save(quarantine_path)

print("Failed records saved to Quarantine")

## Step 7 — PASS → Clean / Trusted Data

Only records that passed the quality gate continue to the trusted layer.

We remove the quality-check columns because they were used for validation, not for the final business table.

In [ ]:
trusted_df = passed_df.drop(
    "completeness_check",
    "validity_check",
    "accuracy_check",
    "uniqueness_check",
    "quality_pass"
)

trusted_df = trusted_df.withColumn(
    "wait_category",
    F.when(F.col("wait_time") < 20, "Short")
     .when(F.col("wait_time") <= 40, "Medium")
     .otherwise("Long")
)

trusted_path = str(LAKEHOUSE_DIR / "appointments_trusted")

trusted_df.write     .format("delta")     .mode("overwrite")     .save(trusted_path)

print("Trusted data saved to Delta Lake")
print("Trusted rows:", trusted_df.count())

## Observation — Quality Gate

We have now demonstrated both required cases:

- **PASS:** valid records continue to the trusted Delta Lake.
- **FAIL:** invalid records are sent to Quarantine.

This prevents low-quality records from entering the trusted data layer.

## Step 8 — Read the Trusted Delta Table

Now we read the trusted data from Delta Lake.

In [ ]:
appointments = spark.read.format("delta").load(trusted_path)

appointments.show(5)
print("Trusted rows:", appointments.count())

## Step 9 — Simple Analytics

Now that the data is clean, we can safely calculate hospital metrics.

In [ ]:
print("Appointments by department:")

appointments.groupBy("department")     .count()     .orderBy(F.desc("count"))     .show()

In [ ]:
print("Average waiting time by department:")

appointments.groupBy("department")     .avg("wait_time")     .orderBy(F.desc("avg(wait_time)"))     .show()

In [ ]:
print("Appointment status:")

appointments.groupBy("status")     .count()     .orderBy(F.desc("count"))     .show()

## Observation 2 — Analytics

Look at the results.

Discuss:

1. Which department has the most appointments?
2. Which department has the highest average waiting time?
3. Which appointment status appears most often?

# Part 2 — Simple RAG Assistant

Now we add a small knowledge base containing hospital appointment policies.

The goal is not to generate new information.

The assistant retrieves the most relevant policy and uses it to answer the question.

In [ ]:
policies = [
    {
        "id": "p1",
        "text": "Patients should arrive 15 minutes before their appointment."
    },
    {
        "id": "p2",
        "text": "Patients can cancel an appointment up to 24 hours before the appointment time."
    },
    {
        "id": "p3",
        "text": "Patients should contact the hospital if they need to change their appointment."
    },
    {
        "id": "p4",
        "text": "Cardiology appointments are available from 8 AM to 4 PM."
    },
    {
        "id": "p5",
        "text": "Dermatology appointments are available from 9 AM to 3 PM."
    }
]

for policy in policies:
    print(policy["id"], ":", policy["text"])

## Step 10 — Create a Simple Retriever

We use BM25 to find the policy that is most related to the user's question.

In [ ]:
from rank_bm25 import BM25Okapi

documents = [policy["text"] for policy in policies]
tokenized_documents = [doc.lower().split() for doc in documents]

bm25 = BM25Okapi(tokenized_documents)

query = "Can I cancel my appointment?"

query_tokens = query.lower().split()
scores = bm25.get_scores(query_tokens)

results = sorted(
    zip(policies, scores),
    key=lambda x: x[1],
    reverse=True
)

for rank, (policy, score) in enumerate(results, start=1):
    print(f"Rank {rank} | {policy['id']} | Score: {score:.4f}")
    print(policy["text"])
    print()

## Observation 3 — Retrieval

Look at the Top 3 results.

- Which policy was retrieved first?
- Is it relevant to the question?
- Why is retrieving the right information important before generating an answer?

## Step 11 — Build the RAG Answer

The retriever gives us the most relevant policy.

We use that policy as the answer source.

In [ ]:
best_policy = results[0][0]

print("Question:", query)
print("Retrieved policy:", best_policy["text"])
print()
print("Answer:", best_policy["text"])

## Step 12 — Try Another Question

Change only the question and run the cell again.

In [ ]:
query = "When should I arrive for my appointment?"

query_tokens = query.lower().split()
scores = bm25.get_scores(query_tokens)

results = sorted(
    zip(policies, scores),
    key=lambda x: x[1],
    reverse=True
)

best_policy = results[0][0]

print("Question:", query)
print("Answer:", best_policy["text"])

## Pair Experiment — Change the Question

Try these questions:

1. `Can I cancel my appointment?`
2. `When should I arrive?`
3. `How can I change my appointment?`
4. `What time is Cardiology available?`
5. `What time is Dermatology available?`

Record the retrieved policy and discuss whether it matches the question.

## Step 13 — Add a Simple Confidence Check

If the retrieval score is too low, we should not pretend that we found a good answer.

In [ ]:
query = "Can I get free parking?"

query_tokens = query.lower().split()
scores = bm25.get_scores(query_tokens)

results = sorted(
    zip(policies, scores),
    key=lambda x: x[1],
    reverse=True
)

best_policy, best_score = results[0]

print("Question:", query)
print("Best score:", round(best_score, 4))

if best_score > 0:
    print("Answer:", best_policy["text"])
else:
    print("I could not find a relevant hospital policy.")

## Observation 4 — Why Confidence Matters

What happens when the question is not covered by the knowledge base?

A RAG system should avoid presenting an unrelated document as a confident answer.

# Part 3 — Final Pipeline Summary

The project has two connected parts:

**Data Engineering**

```text
CSV
 ↓
Spark
 ↓
Data Quality Checks
 ↓
Clean Data
 ↓
Delta Lake
 ↓
Analytics
```

**RAG**

```text
User Question
 ↓
BM25 Retrieval
 ↓
Relevant Hospital Policy
 ↓
Answer
```

## Final Challenge

Change one part of the project.

Choose one:

- Add another hospital policy.
- Add another data quality rule.
- Add another appointment department.
- Ask a new RAG question.

Record:

**What did you change?**

**What happened before?**

**What happened after?**

**Why did the result change?**

## Final Takeaway

Explain the project in your own words:

> I built a hospital appointment data pipeline using Spark. I cleaned the appointment data, applied simple quality rules, stored trusted data in Delta Lake, and created basic analytics. I also built a simple RAG assistant that retrieves hospital policies and uses the relevant policy to answer appointment questions.

# Final Project Architecture

```text
Synthetic CSV Dataset
        |
        v
    Ingestion
        |
        v
      Spark
        |
        v
  Data Quality Checks
        |
        v
   Quality Gate
      /     \
   FAIL     PASS
    |         |
    v         v
Quarantine  Trusted Data
 Delta       |
             v
        Delta Lake
             |
       +-----+------+
       |            |
       v            v
   Analytics      RAG
                    |
                    v
              Hospital FAQ
                    |
                    v
                 Answer
```

# Requirements Checklist

- [x] Simple problem and healthcare domain
- [x] Synthetic dataset
- [x] Data source and ingestion
- [x] Spark processing
- [x] Data quality checks
- [x] Completeness check
- [x] Validity check
- [x] Accuracy / business rule check
- [x] Uniqueness check
- [x] PASS / FAIL quality gate
- [x] FAIL → Quarantine
- [x] PASS → Trusted Delta Lake
- [x] At least one passing and one failing case
- [x] Transformation from raw to trusted data
- [x] Analytics output
- [x] Retrieval-based AI/RAG component
- [x] Experiments and observations
- [x] Architecture shown in the notebook

## End

The main idea is:

**Clean data → Trusted data → Useful analytics → Relevant information → Simple AI answer**